Drude Model Theory: https://www.chem.uci.edu/~lawm/AM%20Ch%201-3.pdf
Plasma constant: http://www.wave-scattering.com/drudefit.html
Real and Imaginary permittivity: https://empossible.net/wp-content/uploads/2020/06/Lecture-Drude-Model-for-Metals.pdf
Free electron density (Cu - Al): https://phys.libretexts.org/Bookshelves/University_Physics/University_Physics_(OpenStax)/University_Physics_III_-_Optics_and_Modern_Physics_(OpenStax)/09%3A_Condensed_Matter_Physics/9.05%3A_Free_Electron_Model_of_Metals


In [1]:
from utils.paths import SAMPLING_DIR, RAW_DIR,PREDICTIONS_DIR

In [2]:
import pandas as pd
import numpy as np
import pickle

In [3]:
import numpy as np

def drude_model_alloy_from_rho(
    ne_pure,    # array: electron densities of pure elements [m^-3]
    rho_alloy,  # float: alloy resistivity [ohm*m]
    wvl,        # wavelength (nm)
    comp,       
    eps_inf=1.0,
    m_eff=9.1093837e-31,
):
    eps0 = 8.8541878128e-12  # F/m
    e = 1.602176634e-19      # C (magnitude)
    c = 299792458.0          # m/s

    comp = np.asarray(comp, dtype=float)
    ne_pure = np.asarray(ne_pure, dtype=float)

    # 1) alloy electron density 
    n_alloy = float(np.dot(ne_pure, comp))  # [m^-3]

    # 2) omega from wavelength
    wvl = np.asarray(wvl, dtype=float)
    wvl_m = wvl * 1e-9
    omega = 2 * np.pi * c / wvl_m

    # 3) Drude parameters 
    omega_p2 = n_alloy * e**2 / (eps0 * m_eff)              # ωp^2 [s^-2]
    tau = m_eff / (n_alloy * e**2 * rho_alloy)              # τ [s]

    # 4) dielectric function parts
    denom = 1.0 + (omega * tau)**2
    e1 = eps_inf - (omega_p2 * tau**2) / denom
    e2 = (omega_p2 * tau) / (omega * denom)

    return e1, e2

elem_cols = ["Cu","Al","Ni"]
ne_pure = [8.47e28,18.1e28,5.64e28]
rho_col = 'resistivity'
wvl = 1550

def row_to_result(row):
    comp = row[elem_cols].to_numpy(dtype=float)      # shape (3,)
    rho_alloy = float(row[rho_col])
    return drude_model_alloy_from_rho(
        ne_pure=ne_pure,
        rho_alloy=rho_alloy,
        wvl=wvl,
        comp=comp,
        eps_inf=1.0,
        m_eff=9.1093837e-31,
        wvl_unit="nm",
    )

In [20]:
comp_space = pd.read_csv(RAW_DIR/"CuNiAl_descriptors.csv")
resistivity = np.load(PREDICTIONS_DIR/'ANN_Predictions/electrical_resistivity_predicted.npy') *1e-8 # resistivity intial units (µΩ·cm)
comp_space['resistivity'] = resistivity

In [16]:
comp_space = pd.read_pickle(PREDICTIONS_DIR/"ANN_Predictions/comp_space_resis_prediction.pkl")
comp_space['resistivity'] = comp_space['resistivity'] *1e-8

In [27]:
comp_space[["e1", "e2"]] = comp_space.apply(
    lambda row: drude_model_alloy_from_rho(
        ne_pure=ne_pure,
        rho_alloy=float(row[rho_col]),
        wvl=wvl,
        comp=row[elem_cols].to_numpy(dtype=float),
    ),
    axis=1,
    result_type="expand"
)

In [28]:
with open(PREDICTIONS_DIR / "ANN_Predictions/ANN_e2_calculated.pkl", "wb") as f:
    pickle.dump(comp_space, f)

Experimental Prediction

In [29]:
experimental_space = pd.read_pickle(PREDICTIONS_DIR/"ANN_Predictions/experimental_resis_prediction.pkl")
experimental_space['resistivity'] = experimental_space['resistivity'] *1e-8

In [31]:
experimental_space[["e1", "e2"]] = experimental_space.apply(
    lambda row: drude_model_alloy_from_rho(
        ne_pure=ne_pure,
        rho_alloy=float(row[rho_col]),
        wvl=wvl,
        comp=row[elem_cols].to_numpy(dtype=float),
    ),
    axis=1,
    result_type="expand"
)

In [32]:
with open(PREDICTIONS_DIR / "ANN_Predictions/ANN_e2_experiment_prediction.pkl", "wb") as f:
    pickle.dump(experimental_space, f)